In [1]:
import json
from pathlib import Path

DATA_DIR = Path("data") / "processed"

with (DATA_DIR / "train.json").open(encoding="utf-8") as file:
    train_documents = json.load(file)

with (DATA_DIR / "validation.json").open(encoding="utf-8") as file:
    valid_documents = json.load(file)

with (DATA_DIR / "vocab.json").open(encoding="utf-8") as file:
    vocab = set(json.load(file))

In [2]:
print("Train kitap sayısı:", len(train_documents))
print("Validation kitap sayısı:", len(valid_documents))
print("Sözlük boyutu:", len(vocab))

print("İlk kitabın kimliği:", train_documents[0]["book_id"])
print("İlk 20 token:", train_documents[0]["tokens"][:20])

Train kitap sayısı: 20
Validation kitap sayısı: 1
Sözlük boyutu: 20000
İlk kitabın kimliği: 57682
İlk 20 token: ['produced', 'by', 'susan', 'skinner', ',', 'chris', '<unk>', ',', '<unk>', 'patten', 'and', 'the', 'online', 'distributed', 'proofreading', 'team', 'at', 'http', ':', '/']


In [3]:
import importlib
import lm.ngram as ngram

importlib.reload(ngram)

bigram_counts, bigram_context_counts = ngram.count_ngrams(
    train_documents, n=2
)

trigram_counts, trigram_context_counts = ngram.count_ngrams(
    train_documents, n=3
)

print("Farklı bigram:", len(bigram_counts))
print("Farklı trigram:", len(trigram_counts))

print("En sık 10 bigram:")
print(bigram_counts.most_common(10))

Farklı bigram: 293055
Farklı trigram: 684112
En sık 10 bigram:
[((',', 'and'), 9937), (('of', 'the'), 8002), (('.', '"'), 5380), (('<unk>', ','), 4806), (('in', 'the'), 4305), (('-', '-'), 4053), ((',', '"'), 3671), (('to', 'the'), 3022), (('.', 'the'), 2895), ((',', 'the'), 2439)]


In [4]:
expected_count = sum(
    len(document["tokens"]) + 1
    for document in train_documents
)

assert sum(bigram_counts.values()) == expected_count
assert sum(trigram_counts.values()) == expected_count

print("Toplam bigram:", sum(bigram_counts.values()))
print("Toplam trigram:", sum(trigram_counts.values()))

Toplam bigram: 1000020
Toplam trigram: 1000020


In [5]:
importlib.reload(ngram)

bigram_probability = ngram.next_word_probability(
    context=("of",),
    word="the",
    ngram_counts=bigram_counts,
    context_counts=bigram_context_counts,
)

trigram_probability = ngram.next_word_probability(
    context=("one", "of"),
    word="the",
    ngram_counts=trigram_counts,
    context_counts=trigram_context_counts,
)

print("P(the | of):", bigram_probability)
print("P(the | one, of):", trigram_probability)

P(the | of): 0.24793951787816818
P(the | one, of): 0.4859154929577465


In [6]:
importlib.reload(ngram)

print("Bigram: 'of' sonrasında")
bigram_predictions = ngram.top_next_words(
    context=("of",),
    ngram_counts=bigram_counts,
    context_counts=bigram_context_counts,
)

for word, probability in bigram_predictions:
    print(f"{word}: %{probability * 100:.2f}")


print("\nTrigram: 'one of' sonrasında")
trigram_predictions = ngram.top_next_words(
    context=("one", "of"),
    ngram_counts=trigram_counts,
    context_counts=trigram_context_counts,
)

for word, probability in trigram_predictions:
    print(f"{word}: %{probability * 100:.2f}")

Bigram: 'of' sonrasında
the: %24.79
a: %4.49
his: %3.60
<unk>: %3.36
this: %1.72

Trigram: 'one of' sonrasında
the: %48.59
them: %8.45
his: %6.87
these: %5.99
those: %4.05


In [7]:
importlib.reload(ngram)

context = ("one", "of")
word = "the"

original = ngram.next_word_probability(
    context, word, trigram_counts, trigram_context_counts
)

smoothed = ngram.smoothed_probability(
    context, word, trigram_counts, trigram_context_counts, vocab
)

print("Smoothing olmadan:", original)
print("Smoothing ile:", smoothed)

Smoothing olmadan: 0.4859154929577465
Smoothing ile: 0.3593927004257868


In [8]:
total_probability = sum(
    ngram.smoothed_probability(
        context,
        candidate,
        trigram_counts,
        trigram_context_counts,
        vocab,
    )
    for candidate in vocab
    if candidate != "<bos>"
)

print("Olasılıkların toplamı:", total_probability)

assert abs(total_probability - 1.0) < 1e-9

Olasılıkların toplamı: 1.0


In [9]:
import lm.evaluation as evaluation

importlib.reload(evaluation)

bigram_ppl = evaluation.perplexity(
    documents=valid_documents,
    n=2,
    ngram_counts=bigram_counts,
    context_counts=bigram_context_counts,
    vocab=vocab,
    k=0.01,
)

trigram_ppl = evaluation.perplexity(
    documents=valid_documents,
    n=3,
    ngram_counts=trigram_counts,
    context_counts=trigram_context_counts,
    vocab=vocab,
    k=0.01,
)

print(f"Bigram validation perplexity: {bigram_ppl:.2f}")
print(f"Trigram validation perplexity: {trigram_ppl:.2f}")

Bigram validation perplexity: 485.25
Trigram validation perplexity: 2952.20


In [11]:
import lm.generation as generation

importlib.reload(generation)

prompt = "it was"

bigram_text = generation.generate_text(
    prompt, 2,
    bigram_counts, bigram_context_counts, vocab,
    k=0.001,
)

trigram_text = generation.generate_text(
    prompt, 3,
    trigram_counts, trigram_context_counts, vocab,
    k=0.001,
)

print("BIGRAM:")
print(bigram_text)

print("\nTRIGRAM:")
print(trigram_text)

BIGRAM:
it was a god and the other. he was the same time to the same time the great deal of their own hands, or three or to have been able

TRIGRAM:
it was a true heart to her. he was not the same, or the other man was killed in an age when it becomes loaded with baggage and horses and
